# **TaniMol: 04 - Butina Clustering**

The primary objective is to identify high-density chemical scaffolds and distinct chemotypes within the dataset. The Butina algorithm, designed specifically for cheminformatics, groups molecules based on a defined Tanimoto similarity threshold. It preserves chemically unique outliers as 'singletons' and assigns actual molecular structures as cluster centroids.

**Input:** Preprocessed activity data and Tanimoto similarity `.npy` matrices   
**Output:** Cluster dictionaries mapping centroids to their members, and singletons list

In [1]:
import numpy as np
import pandas as pd

from src.config import PROCESSED_DIR, CLUSTERING_THRESHOLD, MORGAN_SIM_PATH, MACCS_SIM_PATH, RDKIT_SIM_PATH
from src.clustering import cluster_similarity_matrix, analyze_clusters

### **1. Load Preprocessed Data and Matrices**

Load the standardized biological activity dataset and the precomputed pairwise Tanimoto similarity matrices. These matrices represent the complete structural relationship network for all compounds.


In [2]:
df = pd.read_csv(PROCESSED_DIR / "cleaned_activities.csv")
print(f"Loaded {len(df)} molecules with bioactivity data.")

morgan_sim = np.load(MORGAN_SIM_PATH)
print(f"\n[Morgan] Done. Matrix shape: {morgan_sim.shape}")

maccs_sim = np.load(MACCS_SIM_PATH)
print(f"[MACCS] Done. Matrix shape: {maccs_sim.shape}")

rdkit_sim = np.load(RDKIT_SIM_PATH)
print(f"[RDKit] Done. Matrix shape: {rdkit_sim.shape}")

Loaded 6965 molecules with bioactivity data.

[Morgan] Done. Matrix shape: (6965, 6965)
[MACCS] Done. Matrix shape: (6965, 6965)
[RDKit] Done. Matrix shape: (6965, 6965)


### **2. Implementation of Butina Clustering**

Apply the Butina algorithm across the three diverse molecular representations. A strict Tanimoto similarity cutoff (`THRESHOLD = 0.6`) is enforced to ensure that clustered molecules share a significant proportion of their topological features.

*Threshold value can be adjusted in the cell below*


In [3]:
print(f"Clustering Morgan fingerprints (Tanimoto Threshold: {CLUSTERING_THRESHOLD})...")
morgan_clusters, morgan_singletons = cluster_similarity_matrix(morgan_sim, CLUSTERING_THRESHOLD)
print(f"Found {len(morgan_clusters)} clusters and {len(morgan_singletons)} singletons.")

print(f"\nClustering MACCS keys (Tanimoto Threshold: {CLUSTERING_THRESHOLD})...")
maccs_clusters, maccs_singletons = cluster_similarity_matrix(maccs_sim, CLUSTERING_THRESHOLD)
print(f"Found {len(maccs_clusters)} clusters and {len(maccs_singletons)} singletons.")

print(f"\nClustering RDKit fingerprints (Tanimoto Threshold: {CLUSTERING_THRESHOLD})...")
rdkit_clusters, rdkit_singletons = cluster_similarity_matrix(rdkit_sim, CLUSTERING_THRESHOLD)
print(f"Found {len(rdkit_clusters)} clusters and {len(rdkit_singletons)} singletons.")

Clustering Morgan fingerprints (Tanimoto Threshold: 0.6)...
Found 597 clusters and 527 singletons.

Clustering MACCS keys (Tanimoto Threshold: 0.6)...
Found 73 clusters and 27 singletons.

Clustering RDKit fingerprints (Tanimoto Threshold: 0.6)...
Found 253 clusters and 163 singletons.


### **3. Cluster Analysis**

Extracting key structural statistics from the clustering output provides insight into the chemical diversity of the dataset.

- **Total Clusters:** The number of distinct chemical scaffolds (core structures) identified.
- **Singletons:** Unique molecules that do not share structural similarity with any other compound in the dataset (novel chemotypes).
- **Biggest Cluster Size:** High values indicate extensively explored chemical series, often representing a heavily patented or optimized scaffold.
- **Large Clusters (>50):** Demonstrates the number of major chemical families dominating the dataset.


#### **A. Morgan Fingerprints (ECFP4)**

In [4]:
morgan_stats = analyze_clusters(morgan_clusters, morgan_singletons)


--- Clustering Analysis ---
Total Clusters:           597
Total Singletons:         527
Molecules in Clusters:    6438
Biggest Cluster Size:     183 molecules
Clusters > 50 molecules:  20
---------------------------



#### **B. MACCS Keys**

In [5]:
maccs_stats = analyze_clusters(maccs_clusters, maccs_singletons)


--- Clustering Analysis ---
Total Clusters:           73
Total Singletons:         27
Molecules in Clusters:    6938
Biggest Cluster Size:     4853 molecules
Clusters > 50 molecules:  13
---------------------------



#### **C. RDKit Topological Fingerprints**


In [6]:
rdkit_stats = analyze_clusters(rdkit_clusters, rdkit_singletons)


--- Clustering Analysis ---
Total Clusters:           253
Total Singletons:         163
Molecules in Clusters:    6802
Biggest Cluster Size:     1203 molecules
Clusters > 50 molecules:  29
---------------------------



### **4. Save Clusters**

In [8]:
import pickle
import os

results = {
    "morgan": {
        "similarity_matrix": morgan_sim,
        "clusters": morgan_clusters,
        "singletons": morgan_singletons,
    },
    "maccs": {
        "similarity_matrix": maccs_sim,
        "clusters": maccs_clusters,
        "singletons": maccs_singletons,
    },
    "rdkit": {
        "similarity_matrix": rdkit_sim,
        "clusters": rdkit_clusters,
        "singletons": rdkit_singletons,
    },
}

with open(f"{PROCESSED_DIR}/clustering_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Saved. File size: {os.path.getsize(f'{PROCESSED_DIR}/clustering_results.pkl') / 1e6:.1f} MB")

Saved. File size: 582.2 MB
